# Solutions — Bakehouse Transactions

**Dataset:** `samples.bakehouse.sales_transactions`

**Topics:** aggregation, distinct, date, groupBy

> Reference solutions for practice notebooks.
> **Try the problem yourself first** — only open this when you've made a genuine attempt.
>
> Each solution includes a `# Why:` comment explaining the approach chosen.

In [ ]:
from pyspark.sql import functions as F, types as T
transactions = spark.table("samples.bakehouse.sales_transactions")

---
## Problem 1

In [ ]:
# Why: one agg() call computes both metrics in a single scan of the data.
result_1 = transactions.agg(
    F.count("*").alias("total_transactions"),
    F.round(F.sum("totalPrice"), 2).alias("total_revenue")
)

---
## Problem 2

In [ ]:
# Why: distinct() after select is efficient — only the product column is shuffled.
result_2 = transactions.select("product").distinct().orderBy("product")

---
## Problem 3

In [ ]:
# Why: groupBy + agg lets you compute multiple metrics per group in one step.
result_3 = (
    transactions.groupBy("paymentMethod")
                .agg(
                    F.count("*").alias("transaction_count"),
                    F.round(F.sum("totalPrice"), 2).alias("total_revenue")
                )
)

---
## Problem 4

In [ ]:
# Why: round() prevents floating-point display noise on avg results.
result_4 = (
    transactions.groupBy("product")
                .agg(F.round(F.avg("unitPrice"), 2).alias("avg_unit_price"))
                .orderBy(F.col("avg_unit_price").desc())
)

---
## Problem 5

In [ ]:
# Why: to_date() strips the time component cleanly — date_trunc("day") also works.
result_5 = (
    transactions.withColumn("date", F.to_date("dateTime"))
                .groupBy("date")
                .count()
                .withColumnRenamed("count", "transaction_count")
                .orderBy("date")
)